In [1]:
import pandas as pd
import numpy as np

prices = pd.read_csv(
    "../data/raw/prices.csv",
    index_col=0,
    parse_dates=True
)

returns = prices.pct_change().dropna()

annual_return = returns.mean() * 252
annual_volatility = returns.std() * np.sqrt(252)

cumulative_returns = (1 + returns).cumprod()
rolling_max = cumulative_returns.cummax()
drawdowns = (cumulative_returns - rolling_max) / rolling_max
max_drawdown = drawdowns.min()

risk_summary = pd.DataFrame({
    "Annual Return": annual_return,
    "Annual Volatility": annual_volatility,
    "Max Drawdown": max_drawdown
})

risk_summary["Return / Volatility"] = (
    risk_summary["Annual Return"] / risk_summary["Annual Volatility"]
)

risk_summary

,Annual Return,Annual Volatility,Max Drawdown,Return / Volatility
GLD,0.130166,0.158892,-0.220022,0.819213
IWM,0.116078,0.224685,-0.411333,0.516624
QQQ,0.200348,0.218813,-0.351187,0.915613
SPY,0.144921,0.176898,-0.337173,0.819231
TLT,0.003918,0.149051,-0.483512,0.026289


In [2]:
summary_text = risk_summary.round(4).to_string()

print(summary_text)

     Annual Return  Annual Volatility  Max Drawdown  Return / Volatility
GLD         0.1302             0.1589       -0.2200               0.8192
IWM         0.1161             0.2247       -0.4113               0.5166
QQQ         0.2003             0.2188       -0.3512               0.9156
SPY         0.1449             0.1769       -0.3372               0.8192
TLT         0.0039             0.1491       -0.4835               0.0263


In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv("../.env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [6]:
prompt = f"""
You are a financial risk analyst.

Analyze the following ETF risk summary.

Explain in plain English:
1. Which asset appears riskiest
2. Which asset appears most stable
3. Which asset had the worst drawdown
4. Which asset appears most efficient based on return relative to volatility
5. What this suggests about diversification

Risk Summary:

{summary_text}
"""

In [7]:
response = client.responses.create(
    model="gpt-5.5",
    input=prompt
)

ai_analysis = response.output_text

print(ai_analysis)

1. **Riskiest asset:** **IWM** appears riskiest by day-to-day volatility.  
   - It has the highest annual volatility at **22.47%**, meaning its price fluctuated the most.
   - However, **TLT** also looks risky in a different way because it had very poor returns and the deepest drawdown.

2. **Most stable asset:** **GLD** appears most stable overall.  
   - While **TLT** has the lowest volatility at **14.91%**, it also suffered the worst drawdown.
   - **GLD** had relatively low volatility at **15.89%** and the smallest maximum drawdown at **-22.00%**, making it look more stable in practical terms.

3. **Worst drawdown:** **TLT** had the worst drawdown.  
   - Its maximum drawdown was **-48.35%**, meaning it lost nearly half its value from peak to trough during the period.

4. **Most efficient asset:** **QQQ** appears most efficient based on return relative to volatility.  
   - It had the highest return/volatility ratio at **0.9156**.
   - It also had the highest annual return at **20